# From Research Question to Analysis Strategy

*Notebook #4 in the hands-on MNE series. Assumes the material of notebooks #1 (raw data, filtering, epoching, ERPs), #2 (preprocessing decisions), and #3 (classification with cross-validation, CSP).*

The first three notebooks taught you *how* to process EEG data: how to filter, how to epoch, how to compute an ERP, how to extract features and train a classifier. This notebook asks a different question — **when**.

In practice, a neuroscientist, a clinician, and a BCI engineer may all receive the same EEG recording, yet each will analyse it in an entirely different way. The recording does not dictate the pipeline; the **research question** does. The ability to look at a problem and determine *which* tools are appropriate — before writing a single line of code — is the skill that separates a competent analyst from someone who can merely execute tutorials.

This notebook presents four realistic scenarios built around data you already know: the MNE sample dataset. The data does not change between scenarios. The questions change, and with them, every analytical decision.

> **Pedagogical note.** Each scenario begins with a deliberate pause — a markdown cell asking you to predict which tools and techniques are appropriate *before* seeing the code. These pauses are not decorative. Resist the temptation to scroll past them. The learning happens in the moment of decision, not in the execution that follows.

## Table of contents

1. **The decision framework** — A structured approach to mapping questions onto analysis strategies.
2. **Data preparation** — Loading and minimally preprocessing the MNE sample dataset.
3. **Scenario A: The clinician** — *"Does this patient's auditory cortex respond normally?"* → ERP averaging and component inspection.
4. **Scenario B: The BCI engineer** — *"Can we detect stimulus modality from a single trial?"* → Feature extraction and classification.
5. **Scenario C: The cognitive scientist** — *"When does the brain differentiate auditory from visual input?"* → Temporal decoding.
6. **Scenario D: The oscillation researcher** — *"Does this individual show normal alpha reactivity?"* → Time-frequency decomposition.
7. **Synthesis** — A reference table mapping question types to pipeline choices.
8. **Practice scenarios** — Unsolved problems for self-assessment.

## Position in the textbook

This notebook does not introduce new techniques; it synthesises the tools from notebooks #1–#3 and asks when each is appropriate. It maps loosely onto:

- **Rao Ch. 6 — Building a BCI.** The chapter that moves from individual techniques to system-level design decisions.
- **Rao Ch. 5.1 — Feature Extraction.** The rationale for choosing one feature representation over another.
- **Rao Ch. 9 — BCI Signal Processing.** The taxonomy of brain signals (evoked, induced, oscillatory) and the processing each requires.

## 1. The decision framework

Before any code is written, three questions must be answered. Every EEG analysis pipeline is determined by the answers to these questions.

### Question 1 — What kind of neural signal is informative?

EEG signals fall into a small number of categories, each requiring different extraction methods:

| Signal type | Definition | Key property | Extraction method |
|---|---|---|---|
| **Evoked response (ERP)** | Waveform time-locked *and* phase-locked to a stimulus | Survives averaging across trials | Trial averaging → inspect waveform |
| **Induced oscillation** | Power change time-locked but *not* phase-locked to an event | Survives power averaging, not waveform averaging | Time-frequency decomposition |
| **Ongoing oscillation** | Rhythmic activity not tied to any event | Characterises a brain *state* (e.g., rest vs task) | Spectral analysis of continuous data |

If you cannot determine which of these categories your question targets, you cannot choose a pipeline.

### Question 2 — What temporal granularity is needed?

- **Trial-averaged.** You need a reliable estimate of the typical response. Noise is reduced by averaging dozens or hundreds of repetitions. This is the standard in clinical ERP assessment and cognitive neuroscience.
- **Single-trial.** You need to make a decision about *each individual epoch* — for instance, because you are building a BCI that must respond in real time, or because you want to correlate neural activity with trial-by-trial behaviour.

This distinction fundamentally shapes the pipeline. Averaging is the most powerful denoising tool available; if your question *permits* averaging, you should average. If your question *forbids* it (because each trial needs its own answer), you must rely on spatial filtering, regularisation, and machine learning.

### Question 3 — What constitutes an answer?

- **A waveform or topographic map** to be visually inspected and compared to known norms (clinical/descriptive use).
- **A statistical test** quantifying whether two conditions differ reliably (hypothesis-driven research).
- **A classification accuracy** demonstrating that a machine can distinguish conditions from neural data (BCI engineering, decoding neuroscience).
- **A time course of decodability** revealing *when* information becomes available in the brain (temporal dynamics research).

The form of the answer determines the final analysis step: plotting, statistical inference, cross-validated classification, or temporal generalisation.

---

📖 The remainder of this notebook applies this framework to four scenarios. In each case, we first identify the signal type, the required granularity, and the form of the answer — and only *then* write code.

## 2. Data preparation

All four scenarios use the same dataset — the MNE sample recording from the audio-visual experiment. We load the **unfiltered** version and perform only the minimal preprocessing that any reasonable analysis would require, deferring scenario-specific decisions to each section.

### 2.1 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import mne
from mne.datasets import sample

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

print("MNE:", mne.__version__)

### 2.2 Loading and minimal preprocessing

We load the raw (unfiltered) data, extract events before dropping the stimulus channel, select EEG and EOG channels, apply a broad band-pass filter (0.1–40 Hz), and set the average reference. These are steps that virtually any EEG analysis requires, regardless of the question being asked.

In [ ]:
data_path = sample.data_path()
raw_fname = data_path / "MEG" / "sample" / "sample_audvis_raw.fif"

raw = mne.io.read_raw_fif(raw_fname, preload=True)

# Extract events before dropping the stimulus channel.
events = mne.find_events(raw, stim_channel="STI 014")

# Keep EEG and EOG only.
raw.pick(["eeg", "eog"])

# Broad band-pass filter — appropriate for any downstream analysis below 40 Hz.
raw.filter(l_freq=0.1, h_freq=40.0)

# Set the average reference.
raw.set_eeg_reference("average", projection=True)
raw.apply_proj()

raw

### 2.3 A shared epoch set

All four scenarios examine the contrast between auditory and visual stimuli. We epoch the data once, collapsing left and right within each modality. This shared epoch set is the starting point from which each scenario diverges.

In [ ]:
event_id = {
    "auditory/left":  1,
    "auditory/right": 2,
    "visual/left":    3,
    "visual/right":   4,
}

epochs = mne.Epochs(
    raw, events, event_id,
    tmin=-0.2, tmax=0.5,
    picks="eeg",
    baseline=(None, 0),
    preload=True,
    reject=dict(eeg=100e-6),
)

# Convenience labels: auditory vs visual (ignoring laterality).
epochs_aud = epochs["auditory/left", "auditory/right"]
epochs_vis = epochs["visual/left", "visual/right"]

print(f"Auditory epochs: {len(epochs_aud)}")
print(f"Visual epochs:   {len(epochs_vis)}")

---

The data is now prepared. From this point, **nothing about the data changes.** Only the questions change.

---

## 3. Scenario A — The clinician

### The situation

You are a clinical neurophysiologist. A patient has been referred to your lab with a suspected auditory processing disorder. The referring physician asks:

> *"Does this patient's auditory cortex generate a normal evoked response? Is the N100 component present, and are its latency and amplitude within expected ranges?"*

This is a routine question in clinical neurophysiology. The physician does not need a machine learning model. They need a reliable waveform that they can compare to normative data.

### ❓ Pause — your prediction

Before seeing the code, answer these questions in your head or on paper:

1. What **type of neural signal** does this question target? (Evoked response? Induced oscillation? Ongoing rhythm?)
2. Do you need **single-trial** information, or can you **average**?
3. What will your **output** look like? (A number? A plot? A classification accuracy?)
4. Which MNE functions from notebook #1 will you use?

Think carefully before scrolling.

---

### Framework answers

| Question | Answer | Reasoning |
|---|---|---|
| Signal type | **Evoked response (ERP)** | The N100 is a phase-locked waveform that emerges from averaging. |
| Granularity | **Trial-averaged** | The clinician needs the most reliable estimate possible; individual trial variability is noise. |
| Output | **Waveform + topographic map** | The clinician will inspect the N100 peak latency, amplitude, and scalp distribution and compare them to known norms. |

This is the simplest pipeline: `epochs.average()` → inspect the resulting `Evoked` object.

### Analysis

In [ ]:
# Step 1: Average all auditory trials into a single evoked response.
evoked_aud = epochs_aud.average()

# Step 2: Identify the N100 — the negative peak near 100 ms at fronto-central channels.
# In clinical practice one would select a predefined channel (e.g. Cz or Fz).
fig = evoked_aud.plot(titles="Auditory evoked response — all EEG channels",
                      spatial_colors=True, gfp=True)
plt.show()

In [ ]:
# Step 3: Measure the N100 at a single channel.
# The channel EEG 050 (approximately Fz) is a typical clinical choice for the auditory N100.
ch_name = "EEG 050"
ch_idx = evoked_aud.ch_names.index(ch_name)

# Extract data for this channel in the N100 time window (70–150 ms).
tmin_n100, tmax_n100 = 0.070, 0.150
times = evoked_aud.times
mask = (times >= tmin_n100) & (times <= tmax_n100)

data_ch = evoked_aud.data[ch_idx, :]
peak_idx = np.argmin(data_ch[mask])  # N100 is a negative peak
peak_latency = times[mask][peak_idx] * 1000  # convert to ms
peak_amplitude = data_ch[mask][peak_idx] * 1e6  # convert to µV

print(f"Channel:         {ch_name}")
print(f"N100 latency:    {peak_latency:.1f} ms")
print(f"N100 amplitude:  {peak_amplitude:.2f} µV")
print()
print("Typical normative ranges (healthy adults, auditory oddball):")
print("  Latency:   80–120 ms")
print("  Amplitude: −2 to −8 µV (varies with age, stimulus, and reference)")

In [ ]:
# Step 4: Topographic map at the N100 peak — does it show the expected
# fronto-central distribution?
peak_time = peak_latency / 1000  # back to seconds
fig = evoked_aud.plot_topomap(times=[peak_time], ch_type="eeg",
                               time_unit="s", colorbar=True)
plt.suptitle(f"Topography at N100 peak ({peak_latency:.0f} ms)", y=1.02)
plt.show()

### Interpretation — the clinician's report

A clinician would write something like: *"The auditory N100 was present at fronto-central sites with a latency of ~100 ms and an amplitude within normal limits. The topographic distribution is consistent with bilateral generators in the superior temporal gyrus. No evidence of a grossly abnormal auditory evoked response."*

Note what the clinician **did not** need:
- No machine learning, no cross-validation, no accuracy metric.
- No time-frequency decomposition.
- No single-trial analysis.

The question was descriptive and normative: *is this response present and within range?* The averaging-and-inspection pipeline is the correct and sufficient tool.

---

❓ **Exercise.** Repeat this analysis for the *visual* evoked response. The relevant component is the P100 — a *positive* peak near 100 ms, maximal over occipital channels. Which channel should you select? Should you use `np.argmin` or `np.argmax` to find the peak?

---

## 4. Scenario B — The BCI engineer

### The situation

You are developing a brain-computer interface for a paralysed patient. The system must determine, **on each individual trial**, whether the patient is attending to an auditory or a visual stimulus. This information will be used to drive a communication aid.

> *"Can we classify auditory versus visual trials from a single epoch of EEG?"*

The physician from Scenario A measured the *average* response and compared it to norms. The BCI engineer cannot average — the system must make a decision every few hundred milliseconds, from a single epoch.

### ❓ Pause — your prediction

1. Can you reuse the clinician's pipeline (average → inspect)? Why or why not?
2. What type of output do you need now? (A waveform? A label per trial?)
3. How will you evaluate whether the system works? What metric makes sense?
4. Which techniques from notebook #3 are relevant here?

---

### Framework answers

| Question | Answer | Reasoning |
|---|---|---|
| Signal type | **Evoked response** (same as Scenario A) | The signal is still the ERP — but now we must detect it in a single noisy trial. |
| Granularity | **Single-trial** | The BCI must respond to each epoch individually; averaging is forbidden. |
| Output | **A classification accuracy** (or AUC, kappa) | We need a metric that quantifies how well the system distinguishes the two conditions at the single-trial level. |

The signal is the same; the constraint is different. This is the core lesson: **the same neural phenomenon demands a completely different pipeline when the question changes from "is it present on average?" to "can we detect it on each trial?"**

### Analysis

In [ ]:
from mne.decoding import Vectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [ ]:
# Build the data matrix.
# X: (n_trials, n_channels, n_times) — each trial is a separate observation.
# y: 0 = auditory, 1 = visual.
epochs_clf = epochs["auditory/left", "auditory/right", "visual/left", "visual/right"]
X = epochs_clf.get_data(copy=False)
y_raw = epochs_clf.events[:, -1]
y = np.where(np.isin(y_raw, [1, 2]), 0, 1)

print(f"Data shape: {X.shape}  ({X.shape[0]} trials × {X.shape[1]} channels × {X.shape[2]} time points)")
print(f"Labels:     auditory={np.sum(y==0)}, visual={np.sum(y==1)}")

In [ ]:
# Pipeline: vectorise the epoch → standardise → regularised logistic regression.
# Vectorizer flattens the (channels × times) matrix into a single feature vector.
clf = make_pipeline(
    Vectorizer(),
    StandardScaler(),
    LogisticRegression(solver="liblinear", C=1.0),
)

# Cross-validation: stratified 5-fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")

print(f"Classification accuracy: {scores.mean():.1%} ± {scores.std():.1%}")
print(f"Chance level (balanced classes): 50.0%")

### Interpretation — the engineer's assessment

The classifier discriminates auditory from visual trials at well above chance. This means that the ERP difference visible in the clinician's average is large enough to be detected in individual epochs — a necessary condition for a BCI.

Note the critical contrast with Scenario A:

| Clinician (Scenario A) | BCI engineer (Scenario B) |
|---|---|
| Averaged ~70 auditory trials into one waveform | Classified each trial separately |
| Inspected a plot and compared to norms | Evaluated using cross-validated accuracy |
| Output: "The N100 is present and normal" | Output: "85% single-trial accuracy" |
| No machine learning needed | Machine learning is the central tool |

The underlying neural phenomenon is the same. The questions are different, so the pipelines are different.

---

❓ **Exercise.** The classifier above uses the full epoch (−200 to 500 ms), including the pre-stimulus baseline. Is this a problem? Think about what information from the baseline period could leak into the classifier, and whether this would inflate accuracy in a way that does not generalise to a real BCI.

---

## 5. Scenario C — The cognitive scientist

### The situation

You are a researcher studying the temporal dynamics of multisensory processing. Your question is:

> *"At what point in time after stimulus onset does the brain begin to carry information that distinguishes auditory from visual input? Does this information persist, or does it appear only transiently?"*

Neither the clinician's waveform nor the engineer's single accuracy number answers this question. The clinician's N100 tells you about one predefined time window. The engineer's accuracy collapses the entire epoch into a single scalar. The cognitive scientist needs a **time-resolved** answer: a classification accuracy at every time point.

### ❓ Pause — your prediction

1. How does this question differ from the engineer's question (Scenario B)?
2. If you train a classifier using all time points at once, can you answer this question? Why not?
3. What would you expect the temporal profile to look like? When should accuracy rise above chance?

---

### Framework answers

| Question | Answer | Reasoning |
|---|---|---|
| Signal type | **Evoked response** (still) | The information that distinguishes conditions is carried by the phase-locked ERP. |
| Granularity | **Single-trial** | Decoding operates on individual trials (cross-validated). |
| Output | **A time series of decoding accuracy** | The answer is not a single number but a curve: accuracy as a function of time after stimulus onset. |

The tool is `SlidingEstimator` from `mne.decoding`: a classifier is trained and tested independently at each time point, using only the channel values at that instant.

### Analysis

In [ ]:
from mne.decoding import SlidingEstimator, cross_val_multiscore

In [ ]:
# At each time point, a separate classifier sees only the n_channels values.
estimator = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear", C=1.0),
)

sliding = SlidingEstimator(estimator, scoring="accuracy", n_jobs=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_time = cross_val_multiscore(sliding, X, y, cv=cv)

# scores_time shape: (n_folds, n_times)
mean_scores = scores_time.mean(axis=0)
std_scores = scores_time.std(axis=0)

print(f"Scores shape: {scores_time.shape}  (folds × time points)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

times_epoch = epochs.times
ax.plot(times_epoch * 1000, mean_scores, color="steelblue", linewidth=1.5)
ax.fill_between(times_epoch * 1000,
                mean_scores - std_scores,
                mean_scores + std_scores,
                alpha=0.2, color="steelblue")
ax.axhline(0.5, color="grey", linestyle="--", label="Chance")
ax.axvline(0, color="black", linestyle="-", linewidth=0.8, label="Stimulus onset")

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Accuracy")
ax.set_title("Temporal decoding: when does the brain differentiate auditory from visual?")
ax.legend(loc="upper right")
ax.set_ylim(0.35, 1.0)
plt.tight_layout()
plt.show()

### Interpretation — the scientist's answer

The decoding curve answers the question directly: accuracy is at chance during the baseline, rises sharply after ~60–80 ms (consistent with early cortical processing), peaks around 100–150 ms (the N100/P100 window), and remains above chance for several hundred milliseconds.

This result tells the scientist something that **neither the clinician nor the engineer could provide**:

- The clinician's N100 measurement revealed that the response is present, but nothing about its information content at other time points.
- The engineer's accuracy proved that classification is possible, but collapsed all temporal structure into a single number.
- The temporal decoding curve reveals the *dynamics* — when information appears, how long it persists, and when it fades.

Same data. Different question. Different pipeline. Different insight.

---

❓ **Exercise.** Based on this curve, the engineer from Scenario B might reconsider their pipeline. If most of the discriminative information is concentrated in the 60–200 ms window, what would happen if they restricted their feature vector to that interval? Would accuracy improve (less noise) or drop (less information)? Modify the code in Scenario B to test this.

---

## 6. Scenario D — The oscillation researcher

### The situation

You are a neuroscientist studying resting-state brain rhythms. Your question is:

> *"Does this individual show normal alpha-band (8–12 Hz) reactivity? Specifically, is there evidence of alpha suppression (blocking) during visual stimulus processing?"*

This question has nothing to do with the shape of the evoked waveform. Alpha blocking is an **induced** phenomenon: the *power* in the 8–12 Hz band decreases during visual processing, but this power change is not phase-locked to the stimulus and therefore **vanishes** when you average the raw waveform.

### ❓ Pause — your prediction

1. If you computed `epochs_vis.average()` and looked at the waveform, would you see alpha blocking? Why not?
2. What tool do you need instead? (Hint: you need to extract *power* at each frequency, not the raw waveform.)
3. Over which scalp region would you expect alpha blocking to be strongest for visual stimuli?

---

### Framework answers

| Question | Answer | Reasoning |
|---|---|---|
| Signal type | **Induced oscillation** | Alpha blocking is a power change, not a phase-locked waveform. |
| Granularity | **Power-averaged across trials** | We average the time-frequency *power* (not the raw signal) to obtain a reliable estimate of the spectral change. |
| Output | **A time-frequency map** showing power as a function of time and frequency | The answer is a 2D image: is there a "cold spot" in the alpha band during stimulation? |

The tool is `mne.time_frequency.tfr_morlet` — a wavelet-based decomposition that extracts oscillatory power at each time-frequency point.

### Analysis

We create wider epochs (−0.5 to 1.0 s) to give the wavelet sufficient time resolution at low frequencies, and omit baseline correction in the `Epochs` object so that baseline power can be computed from the time-frequency representation itself.

In [ ]:
from mne.time_frequency import tfr_morlet

In [ ]:
# Wider epochs for time-frequency analysis.
epochs_tf = mne.Epochs(
    raw, events, event_id={"visual/left": 3, "visual/right": 4},
    tmin=-0.5, tmax=1.0,
    picks="eeg",
    baseline=None,        # No baseline correction — we handle this in the TFR
    preload=True,
    reject=dict(eeg=100e-6),
)

print(f"Visual epochs for TF analysis: {len(epochs_tf)}")

In [ ]:
# Compute the time-frequency representation.
freqs = np.arange(4, 30, 1)          # 4–29 Hz in 1 Hz steps
n_cycles = freqs / 2.0               # Adaptive number of cycles (wider window at low freqs)

tfr = tfr_morlet(
    epochs_tf, freqs=freqs, n_cycles=n_cycles,
    return_itc=False, average=True,
)

# Apply baseline correction: express power as percent change relative
# to the pre-stimulus interval (−500 to 0 ms).
tfr.apply_baseline(baseline=(-0.5, 0), mode="percent")

print(f"TFR shape: {tfr.data.shape}  (channels × frequencies × times)")

In [ ]:
# Plot the time-frequency map at a posterior channel (occipital region).
# EEG 059 is approximately Oz in the MNE sample montage.
fig = tfr.plot(["EEG 059"], title="Time-frequency power — visual stimuli (EEG 059 ≈ Oz)",
               combine="mean")
plt.show()

### Interpretation — the oscillation researcher's finding

The time-frequency map should reveal a region of *decreased* power (cool colours) in the 8–12 Hz band following stimulus onset. This is **alpha blocking** — the suppression of the posterior alpha rhythm during visual processing.

Crucially, this phenomenon would be **invisible** in the clinician's ERP analysis (Scenario A). Trial averaging preserves only phase-locked activity. The alpha oscillation is present in each trial, but at a different phase each time, so averaging the raw signal cancels it. Only by computing *power* (which is phase-insensitive) and then averaging across trials can the induced change be observed.

| Analysis | Sees the ERP? | Sees alpha blocking? |
|---|---|---|
| Waveform averaging (Scenarios A, B, C) | ✓ | ✗ |
| Time-frequency power averaging (Scenario D) | Partially | ✓ |

This is the most important conceptual distinction in EEG analysis: **evoked (phase-locked) vs induced (non-phase-locked) activity require fundamentally different extraction methods.** Applying the wrong method does not produce a weak result — it produces no result.

---

❓ **Exercise.** If the clinician from Scenario A had been asked about alpha blocking instead of the N100, their averaging pipeline would have failed completely. Explain in your own words *why* averaging the raw signal eliminates induced oscillatory power. (Hint: consider what happens when you add sine waves of the same frequency but with random phases.)

---

## 7. Synthesis

The table below summarises the four scenarios. Every row used the same data. Every row produced a different — and scientifically valid — result. The difference is entirely in the question.

| Scenario | Role | Question | Signal | Granularity | Tool | Output |
|---|---|---|---|---|---|---|
| A | Clinician | Is the N100 present and normal? | Evoked | Averaged | `epochs.average()` | Waveform + topography |
| B | BCI engineer | Can we classify each trial? | Evoked | Single-trial | `LogisticRegression` + CV | Accuracy (%) |
| C | Cognitive scientist | When does differentiation occur? | Evoked | Single-trial (per time point) | `SlidingEstimator` | Accuracy vs time curve |
| D | Oscillation researcher | Is alpha blocking present? | Induced | Power-averaged | `tfr_morlet` | Time-frequency map |

### The decision flowchart

```
           What signal is informative?
           /                         \
     Phase-locked                Non-phase-locked
     (Evoked / ERP)              (Induced / oscillatory)
          |                              |
    Need single-trial?           Time-frequency
      /          \                decomposition
    No            Yes              (Scenario D)
    |              |
 Average       Need temporal
 & inspect      dynamics?
(Scenario A)    /        \
              No          Yes
              |            |
         Classify      Sliding
         full epoch    estimator
        (Scenario B)  (Scenario C)
```

---

## 8. Practice scenarios

For each scenario below, apply the three-question framework (signal type? granularity? output?) and determine the appropriate analysis strategy. Try to answer before checking the solution.

### Scenario P1

> A sleep researcher records EEG during an overnight session. They want to **track how the dominant frequency changes** from wakefulness (alpha) through light sleep (theta/sigma) to deep sleep (delta). No task or stimulus is involved.

<details>
<summary>Click to reveal the analysis</summary>

- **Signal type:** Ongoing oscillation (no event to time-lock to).
- **Granularity:** Continuous — spectral analysis is computed on sliding windows of the continuous recording, not on epochs.
- **Output:** A spectrogram (power vs frequency vs time) spanning the entire night, or a hypnogram derived from band-power ratios.
- **Tool:** `raw.compute_psd()` with a sliding window, or `mne.time_frequency.psd_array_welch` on successive segments.

This scenario does not involve epoching at all. The analysis operates on the continuous recording.
</details>

### Scenario P2

> A researcher studying attention presents rapid streams of tones. On rare "oddball" trials, a deviant tone is presented. The question: **can a BCI detect whether the user noticed the deviant**, on each trial, to control a spelling interface?

<details>
<summary>Click to reveal the analysis</summary>

- **Signal type:** Evoked response (the P300 to the deviant tone).
- **Granularity:** Single-trial — the BCI must decide on each trial.
- **Output:** Classification accuracy (or information transfer rate).
- **Tool:** xDAWN spatial filtering + classification (as in the upcoming notebook #5 on ERP-BCIs).

This is identical in structure to Scenario B, but the ERP component of interest is the P300 rather than the N100/P100, and the spatial filter of choice is xDAWN (which is optimised for evoked responses) rather than CSP (which is optimised for oscillatory power differences).
</details>

### Scenario P3

> A motor rehabilitation team records EEG while a stroke patient attempts to imagine moving their affected hand. They want to know whether the **mu rhythm (8–13 Hz) over the sensorimotor cortex shows event-related desynchronisation (ERD)** during imagery, as it does in healthy controls.

<details>
<summary>Click to reveal the analysis</summary>

- **Signal type:** Induced oscillation (ERD is a power change, not a phase-locked waveform).
- **Granularity:** Power-averaged across trials — the clinician needs a reliable estimate of whether ERD is present.
- **Output:** Time-frequency map at C3/C4, or a band-power time course showing the mu power dip during imagery.
- **Tool:** `tfr_morlet` or band-pass filtering + Hilbert envelope, as in notebook #3.

Note the parallel with Scenario D: the signal is induced (non-phase-locked), so waveform averaging would fail. The difference from notebook #3 is that here the question is clinical-descriptive ("is ERD present?"), not BCI-oriented ("can we classify left vs right?").
</details>

### Scenario P4

> A developmental psychologist records ERPs from 6-month-old infants listening to speech sounds. Individual infants contribute only 15–20 usable trials (infants are not cooperative subjects). The question: **does the infant brain distinguish native from non-native phonemes?**

<details>
<summary>Click to reveal the analysis</summary>

- **Signal type:** Evoked response (a mismatch negativity or mismatch response to the deviant phoneme).
- **Granularity:** Trial-averaged — with only 15–20 trials the ERP will be noisy, but single-trial classification is impossible with so few examples. Averaging is the only viable option.
- **Output:** A grand-average difference waveform (deviant minus standard) with a statistical test (e.g., cluster-based permutation test across subjects) to assess significance.
- **Tool:** `epochs.average()`, subtraction of conditions, cluster permutation statistics (`mne.stats.permutation_cluster_test`).

This scenario highlights a practical constraint: the number of available trials determines whether single-trial decoding is feasible. With 15 trials per condition, a classifier would have too few training examples to generalise. The averaging approach of Scenario A is the only defensible strategy.
</details>

---

## 9. Key takeaways

1. **The question determines the pipeline, not the data.** The same recording can be analysed in fundamentally different ways depending on what you want to learn.

2. **Averaging is the most powerful tool you have — when the question permits it.** If you can average, you should. If the question demands single-trial decisions (BCI, trial-by-trial correlation), averaging is not available and you must compensate with spatial filtering, regularisation, and machine learning.

3. **Evoked ≠ induced.** This is not a minor technical detail. Applying the wrong extraction method produces no result at all: averaging destroys induced power; spectral analysis obscures the precise temporal structure of the ERP.

4. **The form of the answer matters.** A waveform, a single accuracy number, a temporal accuracy curve, and a time-frequency map are four different kinds of evidence. Each answers a different class of question. Knowing which format matches your question is a prerequisite to choosing the right analysis.

5. **Practical constraints shape the analysis.** The number of available trials, the required response latency, and the clinical vs research context all influence which pipeline is appropriate — independent of what would be theoretically ideal.

---

## 10. What comes next

The remaining notebooks in this series apply these principles to specific BCI paradigms:

- **Notebook #5 — Stimulus-Evoked BCIs.** A deep dive into the P300 speller: the single-trial classification problem from Scenario B, applied to the oddball paradigm, with xDAWN spatial filtering.
- **Notebook #6 — A Complete Pipeline.** A start-to-finish analysis of a full EEG recording, integrating the decision-making framework from this notebook with rigorous preprocessing (ICA, artifact rejection) and interpretation at each stage.

When you encounter a new analysis in those notebooks, ask yourself: *which scenario from this notebook does it resemble?* The answer should tell you why that particular pipeline was chosen.